In [ ]:
# Empty cell required for the "ruff" extension

# Feature model analysis

Feature models are techniques to extract information from data (in this case, a string).
In the context of near-duplicates detection feature models are used by the Simhash algorithm to create fixed-length fingerprints.
Therefore, it's crucial to find the optimal feature model (and its parameters).
In this project, 4 feature models are considered: bag_of_words, shingles, tf-idf and embeddings.

## bag_of_words

The simplest feature model of the four. For a given string, its features are the words it's made of.
Example: "This is a string" -> ("This", "is", "a", "string")
Hence, two strings are "near-equal" iff they contain about the same words, in any order.

It's simple and quick, but cannot detect locality.
Example: "The bank is after the office" is exactly equal to "The office is after the bank"

## shingles

Strings are sliced using a sliding window algorithm, with a certain _window_size_.
Example: "This is a string" -> ("Thi", "his", " is", "is ", ...) with _window_size_=3
As before, two strings are "near-equal" iff they contain about the same shingles, in any order.

Compared to _bag_of_words_, it generates many more elements but it can detect locality (to some degree).
As a middle-ground, the "word-based" shingle feature model is also considered here: it iterates over words, instead of characters.
Example: "This is a string" -> ("This is", "is a", "a string") with _window_size_=2

## tf-idf

A more refined version of _bag_of_words_, which values "rare" words more than "common" ones.
Each word in each string has a weight (0,1).

## embeddings

Generated using the transformer-based model Qwen3-embedding:0.6b.
Two strings are "near-equal" semantically if their embedding is similar.
Despite its great performance, it tends to be computationally more expensive than other models.


In [ ]:
import sys
import pyspark.sql.functions as F
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent))

from src.spark_utils import get_spark
from src.datasets import read_feature_benchmark
from src.near_duplicates import near_duplicates_benchmark
from src.fingerprint import FingerprintParams, create_fingerprints
from src.fingerprint import FINGERPRINT_SIZES

%load_ext autoreload
%autoreload 2

In [ ]:
spark = get_spark()

DATASET_NAME = "hard" # easy / hard
COMPUTE = False
METRIC = "syntax" # semantic / syntax
FEATURE_MODELS = ["tf-idf", "shingles", "bag_of_words", "embeddings"]
F_SIZES = [32, 64, 128, 256, 512] # Must be increasing-order
K_SAMPLE = 3

assert([fs in FINGERPRINT_SIZES for fs in F_SIZES])
assert(DATASET_NAME in ("easy", "hard"))
assert(METRIC in ("semantic", "syntax"))
assert(K_SAMPLE > 0 and K_SAMPLE < F_SIZES[0])
# TODO: assert for feature models

COL_IN1 = "data1"
COL_IN2 = "data2"
COL_OUT1 = "fingerprint1"
COL_OUT2 = "fingerprint2"
TRUTH_COL_SEMANTIC = "is_near_duplicate_semantic"
TRUTH_COL_SYNTAX = "is_near_duplicate_syntax"

datasets = {
    "easy": read_feature_benchmark(which="easy"),
    "hard": read_feature_benchmark(which="hard")
}

dataset = datasets[DATASET_NAME]
dataset.persist()
dataset.count()

def feature_model_benchmark():
    # Iterate over model_feature(shingle_size) and fingeprint_size
    # For each, iterate over K
    methods = FEATURE_MODELS
    fingerprint_sizes = F_SIZES
    experiments = []
    results = []
    shingle_configs = [
        FingerprintParams(shingle_size=3, over_words=False),
        FingerprintParams(shingle_size=5, over_words=False),
        FingerprintParams(shingle_size=7, over_words=False),
        FingerprintParams(shingle_size=3, over_words=True),
        FingerprintParams(shingle_size=5, over_words=True),
        FingerprintParams(shingle_size=7, over_words=True)
    ]
    Ks = [i for i in range(0, fingerprint_sizes[-1], K_SAMPLE)]

    for m, fs in itertools.product(methods, fingerprint_sizes):
        if m == "shingles":
            for conf in shingle_configs:
                experiments.append({
                    "dataset": dataset,
                    "method": m,
                    "size": fs,
                    "other": conf
                })
        else:
            experiments.append({
                "dataset": dataset,
                "method": m,
                "size": fs,
                "other": None
            })

    for exp in tqdm(experiments, desc="Feature model benchmark running..."):
        try:
            df_exp = dataset
            df_exp = create_fingerprints(
                df=df_exp,
                method=exp["method"],
                input_col=COL_IN1,
                output_col=COL_OUT1,
                size=exp["size"],
                other=exp["other"]
            )
            df_exp = create_fingerprints(
                df=df_exp,
                method=exp["method"],
                input_col=COL_IN2,
                output_col=COL_OUT2,
                size=exp["size"],
                other=exp["other"]
            )
            df_exp.persist()
            for K in Ks:
                if K <= exp["size"]:
                    metrics = near_duplicates_benchmark(
                        df=df_exp,
                        f1_col=COL_OUT1,
                        f2_col=COL_OUT2,
                        truth_col=TRUTH_COL_SEMANTIC,
                        K=K
                    )
                    row = {**exp, "K": K, **metrics.__dict__} # TODO: remove dataset column
                    results.append(row)
            df_exp.unpersist()
            
        except Exception as e:
            print(e)

    return pd.DataFrame(results)

In [ ]:
benchmark_df = pd.DataFrame()
if not COMPUTE:
    filename = f"results/featuremodels_{DATASET_NAME}_{METRIC}.csv"
    if Path(filename):
        benchmark_df = pd.read_csv(filename)
    else:
        benchmark_df = feature_model_benchmark()
        benchmark_df.to_csv(filename)
benchmark_df

In [ ]:
# Plots: fingerprint_size to f1_score
# Best f1-scores by model
# Show best f1 score by (model, fingerprint_size)
# Show which K/fingerprint_size ratio has best f1 score by model
# Show top f1 score by (shingle_size, over_words)
top_fs_f1 = benchmark_df.sort_values(by=["size"])[["size", "f1_score"]].groupby(by=["size"]).agg({"f1_score": "max"})
top_fs_f1["f1_score"] = top_fs_f1["f1_score"].round(2)
plt.figure(figsize=(10,6))
plot = sns.barplot(
    data=top_fs_f1,
    x="size",
    y="f1_score",
    hue="f1_score",
    palette="deep"
)
plt.ylim(top_fs_f1["f1_score"].min() - 0.2, 1)
plt.title("Max f1_score for each fingerprint size")
plt.legend(title="Legend", loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

In [ ]:
top_model = benchmark_df.sort_values(by=["f1_score"])[["method", "f1_score"]].groupby(by=["method"]).agg({"f1_score": "max"})
top_model["f1_score"] = top_model["f1_score"].round(2)
plt.figure(figsize=(10,6))
plot = sns.barplot(
    data=top_model,
    x="method",
    y="f1_score",
    hue="f1_score",
    palette="deep"
)
plt.ylim(top_model["f1_score"].min() - 0.2, 1)
plt.title("Best f1_score for each model")
plt.legend(title="Legend", loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

In [ ]:
datasets["easy"].unpersist()
datasets["hard"].unpersist()
spark.stop()